# Furniture_PyTorch_SGD_Dropout

이 노트북은 발표용 재실험을 위해 **한 가지 조건만 명확하게 바꾸고**,  
학습 과정과 결과를 동일한 형식으로 저장하도록 구성했습니다.

## CNN 흐름과 코드 위치
- **[1] 데이터 입력**
- **[2] 특징 추출 - Conv Block 1**
- **[3] 특징 추출 - Conv Block 2**
- **[4] 추가 특징 추출 / 분류**
- **[5] Optimizer 설정**
- **[6] 학습**
- **[7] 성능 평가**
- **[8] 결과 저장**

> Keras와 PyTorch 모두 같은 번호 체계를 사용해 PPT에서 코드 위치를 비교하기 쉽게 만들었습니다.

## 이 실험에서 바뀌는 것

- Dataset: **Furniture**
- Framework: **PyTorch**
- Optimizer: **SGD**
- Version: **Dropout**
- Epoch: **30**
- Dropout: **0.5**
- Spatial Dropout (`Dropout2d`): **사용하지 않음**
- FC Units: **128**

`Baseline → Dropout → SpatialDropout → FC64`

In [ ]:
# 공통 설정
import os, time, random, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# [0] Google Drive 연결 + furniture.zip 압축 해제
#
# 내 드라이브 기준 위치:
# MyDrive/dataset/furniture.zip
#
# 압축 내부가
#   train / val / test
# 이거나
#   furniture/train / furniture/val / furniture/test
# 처럼 한 단계 더 들어 있어도 자동으로 찾아냅니다.
# ------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = Path("/content/drive/MyDrive/dataset/furniture.zip")
EXTRACT_DIR = Path("/content/furniture_dataset")

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"데이터셋 ZIP을 찾을 수 없습니다: {ZIP_PATH}\n"
        "Google Drive의 MyDrive/dataset/furniture.zip 위치를 확인하세요."
    )

# 이전 실행에서 풀어둔 파일이 있으면 지우고 다시 압축 해제
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

shutil.unpack_archive(str(ZIP_PATH), str(EXTRACT_DIR))
print("압축 해제 완료:", EXTRACT_DIR)

# train / val / test 폴더가 모두 들어 있는 실제 데이터 루트 자동 탐색
candidate_roots = [EXTRACT_DIR] + [
    p for p in EXTRACT_DIR.rglob("*") if p.is_dir()
]

valid_roots = [
    p for p in candidate_roots
    if all((p / split).is_dir() for split in ("train", "valid", "test"))
]

if not valid_roots:
    print("\n압축 해제 후 확인된 상위 폴더:")
    for p in sorted(EXTRACT_DIR.iterdir()):
        print(" -", p)
    raise FileNotFoundError(
        "압축 내부에서 train / val / test 폴더를 모두 포함한 위치를 찾지 못했습니다."
    )

# 가장 바깥쪽의 유효한 루트를 사용
DATASET_ROOT = min(valid_roots, key=lambda p: len(p.parts))

print("Dataset ZIP :", ZIP_PATH)
print("Dataset root:", DATASET_ROOT)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
EPOCHS = 30
BATCH_SIZE = 64
IMAGE_SIZE = (128, 128)
NUM_CLASSES = 5
LEARNING_RATE = 0.01
MOMENTUM = 0.9
EXPERIMENT_NAME = "Furniture_PyTorch_SGD_Dropout"

OUTPUT_DIR = Path("/content/experiment_outputs") / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Experiment:", EXPERIMENT_NAME)
print("Dataset root:", DATASET_ROOT)
print("Output:", OUTPUT_DIR)

## [1] 데이터 입력

Keras와 동일하게 `train / val / test` 폴더를 사용하며,  
이미지는 `128×128`, RGB 3채널로 통일합니다.

In [ ]:
# [1] 데이터 입력
EXPECTED_CLASS_NAMES = ['almirah', 'chair', 'fridge', 'table', 'tv']

for split in ["train", "valid", "test"]:
    split_path = DATASET_ROOT / split
    if not split_path.exists():
        raise FileNotFoundError(
            f"{split_path} 폴더가 없습니다. DATASET_ROOT를 실제 Colab 경로로 수정하세요."
        )

transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor()  # [0,255] → [0,1]
])

train_dataset = datasets.ImageFolder(DATASET_ROOT / "train", transform=transform)
val_dataset = datasets.ImageFolder(DATASET_ROOT / "valid", transform=transform)
test_dataset = datasets.ImageFolder(DATASET_ROOT / "test", transform=transform)

if train_dataset.classes != EXPECTED_CLASS_NAMES:
    raise ValueError(
        f"클래스 순서가 예상과 다릅니다.\n"
        f"예상: {EXPECTED_CLASS_NAMES}\n"
        f"실제: {train_dataset.classes}"
    )

CLASS_NAMES = train_dataset.classes

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=generator, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Classes:", CLASS_NAMES)

## [2]~[4] CNN 모델 정의

Keras Furniture 모델과 같은 구조를 사용합니다.  
Keras의 `SpatialDropout2D`에 대응하는 PyTorch Layer는 `Dropout2d`입니다.

In [ ]:
class FurnitureCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()

        # [2] 특징 추출 - Conv Block 1
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

        )

        # [3] 특징 추출 - Conv Block 2
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

        )

        # [4] 추가 특징 추출 - Conv Block 3
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # GAP: Feature Map의 공간 정보를 채널별 평균으로 압축
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # [4] 분류
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

In [ ]:
model = FurnitureCNN(NUM_CLASSES).to(DEVICE)
parameter_count = sum(p.numel() for p in model.parameters())
print(model)
print("Total parameters:", f"{parameter_count:,}")

## [5] Optimizer 설정

- **SGD**
- Learning Rate: **0.01**
- Momentum: **0.9**

In [ ]:
# [5] Optimizer 설정
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)

## [6] 학습

PyTorch에서는 Keras `fit()`에 해당하는 부분을 직접 작성합니다.

`zero_grad → forward → loss → backward → step`

In [ ]:
BEST_MODEL_PATH = OUTPUT_DIR / "best_model.pt"
# [6] 학습
# 30 Epoch를 고정해 학습 양상을 끝까지 관찰합니다.
history_rows = []
best_val_loss = float("inf")
best_epoch = -1

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_count = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # 1) Gradient 초기화
        optimizer.zero_grad()

        # 2) Forward
        logits = model(images)

        # 3) Loss
        loss = criterion(logits, labels)

        # 4) Backward
        loss.backward()

        # 5) Update
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_count += labels.size(0)

    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_count = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_correct += (logits.argmax(dim=1) == labels).sum().item()
            val_count += labels.size(0)

    train_loss = train_loss_sum / train_count
    train_acc = train_correct / train_count
    val_loss = val_loss_sum / val_count
    val_acc = val_correct / val_count

    history_rows.append({
        "epoch": epoch,
        "loss": train_loss,
        "accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), BEST_MODEL_PATH)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

training_seconds = time.time() - start_time
history_df = pd.DataFrame(history_rows).set_index("epoch")

## [7] 성능 평가

In [ ]:
def predict_loader(model, loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            logits = model(images)
            pred = logits.argmax(dim=1).cpu().numpy()
            y_pred.extend(pred)
            y_true.extend(labels.numpy())

    return np.array(y_true), np.array(y_pred)

def evaluate_predictions(y_true, y_pred, prefix):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    report = pd.DataFrame(
        classification_report(
            y_true, y_pred, target_names=CLASS_NAMES,
            output_dict=True, zero_division=0
        )
    ).T
    report.to_csv(OUTPUT_DIR / f"{prefix}_classification_report.csv", encoding="utf-8-sig")

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, xticks_rotation=45, values_format="d"
    )
    ax.set_title(f"{EXPERIMENT_NAME} - {prefix} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{prefix}_confusion_matrix.png", dpi=160, bbox_inches="tight")
    plt.show()

    return {
        f"{prefix}_test_accuracy": acc,
        f"{prefix}_macro_precision": p,
        f"{prefix}_macro_recall": r,
        f"{prefix}_macro_f1": f1
    }

# Final Epoch
final_true, final_pred = predict_loader(model, test_loader)
final_metrics = evaluate_predictions(final_true, final_pred, "final")

# Best Validation Loss
best_model = FurnitureCNN(NUM_CLASSES).to(DEVICE)
best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
best_true, best_pred = predict_loader(best_model, test_loader)
best_metrics = evaluate_predictions(best_true, best_pred, "best")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history_df.index, history_df["accuracy"], label="Train Accuracy")
plt.plot(history_df.index, history_df["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(f"{EXPERIMENT_NAME} - Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "accuracy_curve.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(history_df.index, history_df["loss"], label="Train Loss")
plt.plot(history_df.index, history_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{EXPERIMENT_NAME} - Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "loss_curve.png", dpi=160, bbox_inches="tight")
plt.show()

## [8] 결과 저장

In [ ]:
history_df.to_csv(OUTPUT_DIR / "history.csv", encoding="utf-8-sig")

summary = {
    "experiment": EXPERIMENT_NAME,
    "dataset": "Furniture",
    "framework": "PyTorch",
    "optimizer": "SGD",
    "model_version": "Dropout",
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "momentum": MOMENTUM if "SGD" == "SGD" else np.nan,
    "dropout": 0.5,
    "spatial_dropout_1": np.nan,
    "spatial_dropout_2": np.nan,
    "fc_units": 128,
    "parameter_count": parameter_count,
    "training_seconds": training_seconds,
    "best_epoch_by_val_loss": best_epoch,
    "best_val_loss": float(best_val_loss),
    "best_val_accuracy": float(history_df["val_accuracy"].max()),
    "final_train_accuracy": float(history_df.iloc[-1]["accuracy"]),
    "final_val_accuracy": float(history_df.iloc[-1]["val_accuracy"]),
    "final_train_val_gap": float(history_df.iloc[-1]["accuracy"] - history_df.iloc[-1]["val_accuracy"]),
    "model_definition_code_lines": 36,
    "training_code_lines": 53,
    **final_metrics,
    **best_metrics
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False, encoding="utf-8-sig")
display(summary_df.T)

zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Saved:", zip_path)

## [9] 결과 ZIP 자동 다운로드

Google Colab에서 실행 중이라면, 저장된 실험 결과 ZIP을 브라우저로 자동 다운로드합니다.

In [ ]:
# ------------------------------------------------------------
# [9] Colab에서 결과 ZIP 자동 다운로드
# 학습/평가/저장이 끝난 뒤 생성된 ZIP 파일을 브라우저로 다운로드합니다.
# ------------------------------------------------------------
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Google Colab 환경이 아니므로 자동 다운로드를 실행하지 않았습니다.")
    print("결과 ZIP 경로:", zip_path)